# Лабораторна робота №5. Великі мовні моделі
**Товстоп'ят Олександр, ІО-42**

Локальна Qwen2.5 3B Instruct керує контролером через перевірені Python-функції.
Маршрут, стан правил і час передачі визначає контролер, а не текст LLM.

## Запуск
Python 3.14. Пакети та точні версії наведено в `requirements-lab5.txt` архіву.
Потрібен локальний Ollama HTTP на `127.0.0.1:11435`:
```powershell
$env:OLLAMA_HOST='127.0.0.1:11435'
ollama serve
# в іншому вікні з тим самим OLLAMA_HOST:
ollama pull qwen2.5:3b-instruct-q4_K_M
ollama pull qwen2.5:3b-instruct-q8_0
```
Папки `assets/` і `results/` мають лежати поряд з блокнотом. Ваги LLM
не входять до архіву; точні digest є в результатах вимірювання.
Картинки каналів і CNN ONNX INT8 взято з ЛР4, їх походження та SHA256 — в assets.

`RUN_LIVE=True` повторює вимірювання та всі діалоги. Для перегляду вже
виконаного досліду використовується `False`: це **replay**, а не нове вимірювання.
Збережені авторські оцінки діють лише для діалогів з тим самим hash.
Після нового live-запуску потрібна нова перевірка відповідей.

In [1]:
from pathlib import Path
import json, html
from IPython.display import display, HTML
RUN_LIVE = False
LIVE_SMOKE = True
ASSETS = Path("assets")
RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)
def table(rows):
    if not rows:
        return
    keys = list(rows[0])
    text = "<table><tr>"+"".join("<th>"+html.escape(str(k))+"</th>" for k in keys)+"</tr>"
    for row in rows:
        text += "<tr>"+"".join("<td>"+html.escape(str(row.get(k,"")))+"</td>" for k in keys)+"</tr>"
    display(HTML(text+"</table>"))
def raw(title,value):
    display(HTML("<details><summary>"+html.escape(title)+"</summary><pre>"+
                 html.escape(json.dumps(value,ensure_ascii=False,indent=2))+"</pre></details>"))


## 1. Мережевий контролер
25 вузлів, seed9, 8 вилучених ребер; місткості seed42. CNN читає цифри
на картинках: 2–9 означають 200–900 Мбіт/с. Контролер відкриває мережу
локально через LSA. Оператор може явно запитати telemetry окремого вузла,
але `get_topology` повертає лише KB, не всю фізичну мережу.

Час передачі 1 ГБ (10^9 байт):
`T = sum(delay_ms)/1000 + 8*size_bytes/(minMbps*10^6)`.
Для незавершеного префікса transfer_seconds=null. Пошук перебирає пороги
місткості та найменшу суму затримок; це оптимум лише відомого графа.
Повідомлення LSA синхронізовані в моделі, реальний обмін пакетами не імітується.

### base

In [2]:
"""Незмінені визначення ЛР2/3 для самодостатньої ЛР4."""
import math
import random
from dataclasses import dataclass
import networkx as nx

def generate_graph_topology(num_nodes=25, edges_to_remove=5, weight_min=1, weight_max=10, seed=None):
    """Генератор топології з ЛР1; додано параметр seed для відтворюваності експериментів."""
    rng = random.Random(seed)
    G = nx.Graph()

    cols = math.ceil(math.sqrt(num_nodes))
    rows = math.ceil(num_nodes / cols)

    node_count = 0
    for r in range(rows):
        for c in range(cols):
            if node_count >= num_nodes:
                break
            G.add_node((r, c), address=f"10.0.{r}.{c}", pos=(c, -r))
            node_count += 1

    for r in range(rows):
        for c in range(cols):
            if not G.has_node((r, c)):
                continue
            if G.has_node((r, c + 1)):
                G.add_edge((r, c), (r, c + 1), weight=rng.randint(weight_min, weight_max))
            if G.has_node((r + 1, c)):
                G.add_edge((r, c), (r + 1, c), weight=rng.randint(weight_min, weight_max))

    removed_count, attempts = 0, 0
    while removed_count < edges_to_remove and attempts < 1000:
        edges = list(G.edges())
        if not edges:
            break
        u, v = rng.choice(edges)
        weight = G[u][v]['weight']
        G.remove_edge(u, v)
        if nx.is_connected(G):
            removed_count += 1
        else:
            G.add_edge(u, v, weight=weight)
        attempts += 1

    if removed_count < edges_to_remove:
        print(f"Увага: вдалося видалити лише {removed_count} ребер без втрати зв'язності.")
    return G


@dataclass
class LinkStateAdvertisement:
    origin: str
    version: int
    links: dict


class KnowledgeBase:
    """TELL: повні версійовані таблиці; ASK використовує похідний граф."""

    def __init__(self):
        self.records = {}

    def tell(self, advertisements):
        changed = []
        for ad in advertisements:
            old = self.records.get(ad.origin)
            if old is None or ad.version > old.version:
                self.records[ad.origin] = LinkStateAdvertisement(
                    ad.origin, ad.version, dict(ad.links))
                changed.append(ad.origin)
        return changed

    def graph(self):
        graph = nx.Graph()
        graph.add_nodes_from(sorted(self.records))
        for origin, ad in sorted(self.records.items()):
            for neighbor, weight in sorted(ad.links.items()):
                other = self.records.get(neighbor)
                # Новіша повна таблиця може явно спростувати старе ребро.
                if other is not None and origin not in other.links:
                    continue
                graph.add_edge(origin, neighbor, weight=weight)
        return graph


class KnowledgeDataPlane:
    """Локальні таблиці та кеші анонсів лише безпосередніх сусідів.

    Початкові кеші синхронізовані; подія зміни атомарно оновлює обидва кінці.
    Це навчальна модель анонсів, а не симулятор протоколу OSPF.
    """

    def __init__(self, G, src, dst):
        self.G = G.copy()
        self._node = {data["address"]: node for node, data in self.G.nodes(data=True)}
        if src not in self._node or dst not in self._node:
            raise ValueError("Невідома адреса відправника або отримувача")
        if any(not math.isfinite(d["weight"]) or d["weight"] <= 0
               for _, _, d in self.G.edges(data=True)):
            raise ValueError("Затримки мають бути скінченними додатними числами")
        self.versions = {address: 1 for address in self._node}
        self.head = src
        self.rules = []

    def _announcement(self, address):
        node = self._node[address]
        links = {self.G.nodes[nb]["address"]: self.G[node][nb]["weight"]
                 for nb in sorted(self.G.neighbors(node))}
        return LinkStateAdvertisement(address, self.versions[address], links)

    def observe(self):
        own = self._announcement(self.head)
        return self.head, [own] + [self._announcement(nb) for nb in sorted(own.links)]

    def install_rule(self, next_addr):
        if next_addr not in self._announcement(self.head).links:
            raise ValueError("Неможливо встановити правило через відсутній канал")
        self.rules.append((self.head, next_addr))
        self.head = next_addr

    def revoke_last_rule(self):
        # Площина керування, а не фізичне пересилання пакета назад.
        self.head, _ = self.rules.pop()

    def change_link(self, u, v, weight=None):
        n_u, n_v = self._node[u], self._node[v]
        if not self.G.has_edge(n_u, n_v):
            raise ValueError("Оновлюваного каналу немає")
        if weight is None:
            self.G.remove_edge(n_u, n_v)
        else:
            if not math.isfinite(weight) or weight <= 0:
                raise ValueError("Затримка має бути скінченною і додатною")
            self.G[n_u][n_v]["weight"] = weight
        self.versions[u] += 1
        self.versions[v] += 1
        return [self._announcement(u), self._announcement(v)]


class KnowledgeController:
    """TELL → ASK → одна дія; без доступу до топології середовища."""

    def __init__(self, src, dst):
        self.src, self.dst = src, dst
        self.kb = KnowledgeBase()
        self.path = [src]
        self.installed = self.revoked = 0
        self.log, self.frames = [], []
        self.success = False

    @staticmethod
    def _xy(address):
        return tuple(map(int, address.split(".")[-2:]))

    def ask(self):
        graph = self.kb.graph()
        if self.src not in graph:
            return None
        distances, paths = nx.single_source_dijkstra(graph, self.src, weight="weight")
        if self.dst in paths:
            return paths[self.dst]
        frontier = [node for node in paths if node not in self.kb.records]
        if not frontier:
            return None
        avg = sum(d["weight"] for _, _, d in graph.edges(data=True)) / max(1, graph.number_of_edges())
        dst_xy = self._xy(self.dst)

        def score(node):
            h = sum(abs(a - b) for a, b in zip(self._xy(node), dst_xy))
            return distances[node] + avg * h, distances[node], node

        return paths[min(frontier, key=score)]

    @property
    def delay(self):
        graph = self.kb.graph()
        return sum(graph[u][v]["weight"] for u, v in zip(self.path, self.path[1:]))

    def summary(self):
        return dict(nodes=len(self.path), delay=self.delay,
                    installed=self.installed, revoked=self.revoked)

    def _record(self, step, text, verbose=False):
        line = f"{step:03d} | {text}"
        self.log.append(line)
        if verbose:
            print(line)

    def _tell(self, ads, step, verbose, source):
        ads = list(ads)
        changed = self.kb.tell(ads)
        versions = ", ".join(f"{a.origin}:v{a.version}" for a in ads if a.origin in changed)
        self._record(step, f"TELL ({source}): {versions or 'без нових фактів'}", verbose)

    def run(self, dp, verbose=False, max_steps=10_000, event=None):
        for step in range(1, max_steps + 1):
            if event is not None:
                ads = event(dp, self, step)
                if ads:
                    self._record(step, "ПОДІЯ: зовнішнє оновлення стану каналів", verbose)
                    self._tell(ads, step, verbose, "подія")
            head, ads = dp.observe()
            if head != self.path[-1]:
                raise RuntimeError("Стан середовища не відповідає шляху контролера")
            self._tell(ads, step, verbose, "спостереження")
            plan = self.ask()
            self._record(step, "ASK: " + (" → ".join(plan) if plan else "маршрутів/фронтирів немає"), verbose)
            if plan is None and len(self.path) > 1:
                # Навіть без обхідного шляху прибираємо правила, а не лишаємо
                # недійсний префікс. Одна операція керування за крок.
                plan = [self.src]

            common = 0
            if plan is not None:
                while common < min(len(plan), len(self.path)) and plan[common] == self.path[common]:
                    common += 1
            if plan is None:
                # Вичерпано всі досяжні фронтири; явна невдача.
                self._record(step, "ЗАВЕРШЕННЯ: отримувач недосяжний за вичерпаними знаннями", verbose)
                return False
            if common < len(self.path):
                u, v = self.path[-2:]
                dp.revoke_last_rule()
                self.path.pop()
                self.revoked += 1
                action = f"ВІДКЛИКАТИ {u} → {v}"
            elif head == self.dst:
                self.success = True
                action = "ЗАВЕРШЕННЯ: чинний шлях до отримувача"
            else:
                next_addr = plan[len(self.path)]
                dp.install_rule(next_addr)
                self.path.append(next_addr)
                self.installed += 1
                action = f"ВСТАНОВИТИ {head} → {next_addr}"
            self._record(step, action, verbose)
            known = self.kb.graph()
            self.frames.append(dict(step=step, head=self.path[-1], path=list(self.path),
                                    known=list(known.edges()),
                                    weights={(u, v): d["weight"] for u, v, d in known.edges(data=True)},
                                    short=action))
            if self.success:
                return True
        self._record(max_steps, "ЗАВЕРШЕННЯ: перевищено ліміт кроків", verbose)
        return False



"""CPU inference контролера: без навчального PyTorch."""
import time
import numpy as np
import onnxruntime as ort


class Int8Recognizer:
    def __init__(self, model_path):
        options = ort.SessionOptions()
        options.intra_op_num_threads = 1
        options.inter_op_num_threads = 1
        self.session = ort.InferenceSession(str(model_path), options,
                                            providers=["CPUExecutionProvider"])
        self.cache = {}
        self.calls = self.cache_hits = 0
        self.session_seconds = self.total_seconds = 0.0

    def recognize(self, u, v, image):
        start = time.perf_counter_ns()
        image = np.asarray(image)
        if image.dtype != np.uint8 or image.shape != (28, 28):
            raise ValueError("Маркування має бути uint8 зображенням 28×28")
        key = tuple(sorted((u, v)))
        if key in self.cache:
            self.cache_hits += 1
            return self.cache[key]
        images = np.ascontiguousarray(image.reshape(1, 1, 28, 28), dtype=np.float32) / 255.0
        feed = {"images": images}
        inference_start = time.perf_counter_ns()
        logits = self.session.run(["logits"], feed)[0]
        self.session_seconds += (time.perf_counter_ns() - inference_start) / 1e9
        capacity = (int(logits.argmax(axis=1)[0]) + 2) * 100
        self.cache[key] = capacity
        self.calls += 1
        self.total_seconds += (time.perf_counter_ns() - start) / 1e9
        return capacity

"""Час конвеєрної передачі та точний пошук у переданому відомому графі."""
import heapq
import math


def path_metrics(graph, path, size_bytes=10**9):
    if not math.isfinite(size_bytes) or size_bytes < 0:
        raise ValueError("Розмір потоку має бути скінченним і невід'ємним")
    edges = list(zip(path, path[1:]))
    delays, capacities = [], []
    for u, v in edges:
        delay, capacity = graph[u][v]["weight"], graph[u][v]["capacity"]
        if not math.isfinite(delay) or delay < 0:
            raise ValueError("Неприпустима затримка")
        if not math.isfinite(capacity) or capacity <= 0:
            raise ValueError("Пропускна здатність має бути додатною")
        delays.append(delay)
        capacities.append(capacity)
    bottleneck = min(capacities) if capacities else None
    payload = 8 * size_bytes / (bottleneck * 1e6) if bottleneck else 0.0
    return dict(delay_ms=sum(delays), bottleneck_mbps=bottleneck,
                bottleneck_edges=[edge for edge, cap in zip(edges, capacities)
                                  if cap == bottleneck],
                payload_seconds=payload, total_seconds=sum(delays) / 1000 + payload)


def best_transfer_path(graph, src, dst, size_bytes=10**9):
    path_metrics(graph, [], size_bytes)
    if src not in graph or dst not in graph:
        return None
    if src == dst:
        return [src]
    for u, v in graph.edges:
        path_metrics(graph, [u, v], size_bytes)
    candidates = []
    for threshold in sorted({attrs["capacity"] for _, _, attrs in graph.edges(data=True)}):
        initial = (0, 1, (src,))
        best = {src: initial}
        heap = [(initial, src)]
        while heap:
            score, node = heapq.heappop(heap)
            if best.get(node) != score:
                continue
            if node == dst:
                path = list(score[2])
                metrics = path_metrics(graph, path, size_bytes)
                candidates.append((metrics["total_seconds"], len(path), tuple(path)))
                break
            for neighbor in sorted(graph.neighbors(node)):
                attributes = graph[node][neighbor]
                if attributes["capacity"] < threshold or neighbor in score[2]:
                    continue
                new_score = (score[0] + attributes["weight"], score[1] + 1,
                             score[2] + (neighbor,))
                if neighbor not in best or new_score < best[neighbor]:
                    best[neighbor] = new_score
                    heapq.heappush(heap, (new_score, neighbor))
    return list(min(candidates)[2]) if candidates else None

"""Агент отримує затримки через LSA, картинки — через інвентаризацію."""
import random
import networkx as nx
import numpy as np


class BandwidthDataPlane(KnowledgeDataPlane):
    def __init__(self, G, src, dst, test_images, test_digits, capacity_seed=42):
        super().__init__(G, src, dst)
        rng = random.Random(capacity_seed)
        self._inventory = {}
        self._allowed_edges = set()
        pools = {digit: np.flatnonzero(test_digits == digit).tolist() for digit in range(2, 10)}
        if any(not pool for pool in pools.values()):
            raise ValueError("Для інвентаризації потрібні зображення всіх цифр 2–9")
        edges = sorted((tuple(sorted((self.G.nodes[u]["address"], self.G.nodes[v]["address"]))),
                        u, v) for u, v in self.G.edges)
        for key, u, v in edges:
            digit = rng.randint(2, 9)
            image_index = rng.choice(pools[digit])
            self.G[u][v].update(capacity=digit * 100, digit=digit, image_index=image_index)
            self._inventory[key] = np.asarray(test_images[image_index]).copy()

    def observe(self):
        head, advertisements = super().observe()
        for ad in advertisements:
            self._allowed_edges.update(tuple(sorted((ad.origin, neighbor))) for neighbor in ad.links)
        return head, advertisements

    def image(self, u, v):
        key = tuple(sorted((u, v)))
        if key not in self._allowed_edges:
            raise PermissionError("Інвентаризація доступна лише для вже оголошених каналів")
        return self._inventory[key].copy()


class BandwidthController(KnowledgeController):
    def __init__(self, src, dst, recognizer, size_bytes=10**9):
        super().__init__(src, dst)
        path_metrics(nx.Graph(), [], size_bytes)
        self.recognizer = recognizer
        self.size_bytes = size_bytes
        self.graph_with_capacities = nx.Graph()
        self._image = None

    def run(self, dp, verbose=False, max_steps=10_000):
        self._image = dp.image
        return super().run(dp, verbose=verbose, max_steps=max_steps)

    def ask(self):
        graph = self.kb.graph()
        for u, v in sorted(graph.edges()):
            key = tuple(sorted((u, v)))
            if key not in self.recognizer.cache:
                capacity = self.recognizer.recognize(u, v, self._image(u, v))
                self._record(len(self.frames) + 1,
                    f"CNN: {u} ↔ {v}: {capacity // 100}, {capacity} Мбіт/с")
            else:
                # Кеш читається без повторної інвентаризації та session.run.
                capacity = self.recognizer.cache[key]
                self.recognizer.cache_hits += 1
            graph[u][v]["capacity"] = capacity
        self.graph_with_capacities = graph
        path = best_transfer_path(graph, self.src, self.dst, self.size_bytes)
        if path is not None:
            return path
        if self.src not in graph:
            return None
        reachable = nx.node_connected_component(graph, self.src)
        frontier = sorted(node for node in reachable if node not in self.kb.records)
        average_delay = sum(d["weight"] for _, _, d in graph.edges(data=True)) / max(1, graph.number_of_edges())
        candidates = []
        for node in frontier:
            route = best_transfer_path(graph, self.src, node, self.size_bytes)
            if route is None:
                continue
            heuristic = sum(abs(a - b) for a, b in zip(self._xy(node), self._xy(self.dst)))
            score = path_metrics(graph, route, self.size_bytes)["total_seconds"] + average_delay * heuristic / 1000
            candidates.append((score, len(route), node, tuple(route)))
        return list(min(candidates)[3]) if candidates else None

### network

In [3]:
"""One flow; deterministic controller owns every network mutation."""
import json
from pathlib import Path
import networkx as nx
import numpy as np

DIRECTIONS = {"N":(-1,0),"E":(0,1),"S":(1,0),"W":(0,-1)}


class ImageDataPlane(KnowledgeDataPlane):
    def __init__(self, graph, src, dst, inventory):
        super().__init__(graph,src,dst)
        self.inventory = inventory

    def image(self, a, b):
        return self.inventory[tuple(sorted((a,b)))].copy()

    def telemetry(self, node):
        own = self._announcement(node)
        return [own]+[self._announcement(nb) for nb in sorted(own.links)]


class AvoidController(BandwidthController):
    def __init__(self,src,dst,recognizer,avoid,size_bytes):
        super().__init__(src,dst,recognizer,size_bytes)
        self.avoid = set(avoid)

    def ask(self):
        # Parent recognizes all known edges. Only planning sees the filtered view.
        original = self.kb
        allowed = type(original)()
        allowed.records = {a:type(ad)(ad.origin,ad.version,
            {b:w for b,w in ad.links.items() if b not in self.avoid})
            for a,ad in original.records.items() if a not in self.avoid}
        self.kb = allowed
        try:
            return super().ask()
        finally:
            self.kb = original


class NetworkFacade:
    def __init__(self,assets_dir,seed=9,size_bytes=10**9):
        assets = Path(assets_dir)
        data = json.loads((assets/"network.json").read_text(encoding="utf-8"))
        if seed != data["seed"]:
            raise ValueError("Цей набір фіксованих картинок відповідає seed9")
        images = np.load(assets/"images.npz")["images"]
        graph, inventory = nx.Graph(), {}
        for node in data["nodes"]:
            graph.add_node(tuple(map(int,node["address"].split(".")[-2:])),
                           address=node["address"],pos=node["pos"])
        for d,image in zip(data["edges"],images):
            a,b = [tuple(map(int,x.split(".")[-2:])) for x in (d["a"],d["b"])]
            graph.add_edge(a,b,weight=d["delay_ms"],capacity=d["true_mbps"])
            inventory[tuple(sorted((d["a"],d["b"])))] = image
        self.size_bytes = size_bytes
        self.dp = ImageDataPlane(graph,"10.0.0.0","10.0.4.4",inventory)
        self.recognizer = Int8Recognizer(assets/"model_int8.onnx")
        self.controller = AvoidController("10.0.0.0","10.0.4.4",self.recognizer,[],size_bytes)
        self.controller._image = self.dp.image

    def _node(self,node):
        if not isinstance(node,str) or node not in self.dp._node:
            raise ValueError(f"Невідомий вузол: {node}")

    def _observe(self,node):
        self.controller.kb.tell(self.dp.telemetry(node))
        # Recognize for telemetry without invoking routing or mutating rules.
        for a,b in sorted(self.controller.kb.graph().edges()):
            key = tuple(sorted((a,b)))
            if key not in self.recognizer.cache:
                self.recognizer.recognize(a,b,self.dp.image(a,b))

    def _graph(self):
        graph = self.controller.kb.graph()
        for a,b in list(graph.edges()):
            cap = self.recognizer.cache.get(tuple(sorted((a,b))))
            if cap is None:
                graph.remove_edge(a,b)
            else:
                graph[a][b]["capacity"] = cap
        return graph

    def state(self):
        c = self.controller
        graph = self._graph()
        valid = all(graph.has_edge(a,b) for a,b in zip(c.path,c.path[1:]))
        metrics = path_metrics(graph,c.path,self.size_bytes) if valid else None
        complete = valid and c.path[-1] == c.dst
        return dict(src=c.src,head=self.dp.head,dst=c.dst,path=list(c.path),
                    rules=[dict(a=a,b=b) for a,b in self.dp.rules],
                    complete=complete,valid=valid,avoid=sorted(c.avoid),
                    size_bytes=self.size_bytes,installed=c.installed,revoked=c.revoked,
                    delay_ms=metrics["delay_ms"] if metrics else None,
                    bottleneck_mbps=metrics["bottleneck_mbps"] if metrics else None,
                    bottleneck_edges=metrics["bottleneck_edges"] if metrics else [],
                    transfer_seconds=metrics["total_seconds"] if complete else None,
                    prefix_seconds=metrics["total_seconds"] if metrics else None,
                    capacity_source="CNN ONNX INT8 prediction")

    def topology(self):
        graph = self._graph()
        return dict(nodes=sorted(graph.nodes),
                    links=[dict(a=a,b=b,delay_ms=d["weight"],
                        recognized_capacity_mbps=d["capacity"])
                        for a,b,d in sorted(graph.edges(data=True))])

    @staticmethod
    def _direction(a,b):
        ra,ca = map(int,a.split(".")[-2:])
        rb,cb = map(int,b.split(".")[-2:])
        return next((k for k,v in DIRECTIONS.items() if v==(rb-ra,cb-ca)),None)

    def link_stats(self,node):
        self._node(node)
        self._observe(node)
        ad = self.controller.kb.records[node]
        return dict(node=node,links=[dict(interface=self._direction(node,nb),
            neighbor=nb,delay_ms=w,
            recognized_capacity_mbps=self.recognizer.cache[tuple(sorted((node,nb)))])
            for nb,w in sorted(ad.links.items())])

    def install_rule(self,interface):
        c = self.controller
        if interface=="REVOKE":
            if not self.dp.rules:
                raise ValueError("Немає правил для відкликання")
            a,b = self.dp.rules[-1]
            self.dp.revoke_last_rule()
            c.path.pop()
            c.revoked += 1
            c._record(len(c.log)+1,f"ВІДКЛИКАТИ {a} → {b}")
        else:
            if interface not in DIRECTIONS:
                raise ValueError("Інтерфейс має бути N/E/S/W/REVOKE")
            # Validate against environment before changing even the KB.
            links = self.dp._announcement(self.dp.head).links
            nb = next((b for b in links if self._direction(self.dp.head,b)==interface),None)
            if nb is None or nb in c.path or nb in c.avoid:
                raise ValueError("Відсутній/заборонений інтерфейс або цикл")
            self._observe(self.dp.head)
            a = self.dp.head
            self.dp.install_rule(nb)
            c.path.append(nb)
            c.installed += 1
            c._record(len(c.log)+1,f"ВСТАНОВИТИ {a} → {nb}")
        c.success = c.path[-1]==c.dst
        return self.state()

    def install_path(self,dst,src=None,avoid=None):
        src = self.controller.src if src is None else src
        avoid = [] if avoid is None else avoid
        self._node(src)
        self._node(dst)
        if not isinstance(avoid,list) or any(not isinstance(a,str) for a in avoid):
            raise ValueError("avoid має бути списком адрес")
        for node in avoid:
            self._node(node)
        if src in avoid or dst in avoid:
            raise ValueError("Відправника/отримувача не можна оминути")
        old = self.controller
        while self.dp.rules:
            self.install_rule("REVOKE")
        kb, log, installed, revoked = old.kb, list(old.log), old.installed, old.revoked
        self.dp.head, self.dp.dst = src,dst
        c = self.controller = AvoidController(src,dst,self.recognizer,avoid,self.size_bytes)
        c.kb,c.log,c.installed,c.revoked = kb,log,installed,revoked
        success = c.run(self.dp)
        return dict(success=success,**self.state(),controller_log=list(c.log))

    def fail_link(self,a,b):
        self._node(a)
        self._node(b)
        ads = self.dp.change_link(a,b)
        c = self.controller
        c._tell(ads,len(c.log)+1,False,"відмова")
        edge = frozenset((a,b))
        while any(frozenset(pair)==edge for pair in self.dp.rules):
            self.install_rule("REVOKE")
        c.success = c.path[-1]==c.dst
        return dict(failed=[a,b],**self.state())

    def oracle(self,avoid=None):
        graph = nx.relabel_nodes(self.dp.G,{n:d["address"] for n,d in self.dp.G.nodes(data=True)})
        graph.remove_nodes_from(avoid or [])
        path = best_transfer_path(graph,self.controller.src,self.controller.dst,self.size_bytes)
        return dict(path=path,metrics=path_metrics(graph,path,self.size_bytes) if path else None,
                    source="evaluator-only full true topology")

### tools

In [4]:
"""Six JSON-schema tools; no arbitrary method dispatch."""
import jsonschema


def tool(name,description,properties=None,required=None):
    return dict(type="function",function=dict(name=name,description=description,
        parameters=dict(type="object",properties=properties or {},
                        required=required or [],additionalProperties=False)))


ADDRESS = dict(type="string",description="Адреса вузла, наприклад 10.0.0.0")
TOOL_SCHEMAS = [
    tool("get_flow_state","Поточний head/dst, правила, complete, delay_ms, час останнього завершеного потоку transfer_seconds і всі bottleneck_edges. Для неповного шляху transfer_seconds=null."),
    tool("get_link_stats","Телеметрія вказаного вузла: напрямки, сусіди, затримки мс, розпізнані CNN місткості Мбіт/с.",{"node":ADDRESS},["node"]),
    tool("install_rule","Одна дія: N північ, E схід, S південь, W захід, REVOKE відкликати останнє правило.",
         {"interface":dict(type="string",enum=["N","E","S","W","REVOKE"])},["interface"]),
    tool("install_path","Побудувати й встановити повний шлях детермінованим контролером. src за замовчуванням поточний відправник. avoid — вузли для обходу. Повертає path, час/вузькі місця та журнал.",
         {"src":ADDRESS,"dst":ADDRESS,"avoid":dict(type="array",items=ADDRESS,uniqueItems=True)},["dst"]),
    tool("get_topology","Повернути тільки вже відомі контролеру вузли/канали, без сканування мережі."),
    tool("simulate_link_failure","Емулювати відмову існуючого каналу. Прибирає недійсні правила, але НЕ перебудовує шлях; потім потрібен install_path.",
         {"a":ADDRESS,"b":ADDRESS},["a","b"]),
]


class ToolDispatcher:
    def __init__(self,network):
        self.network = network
        self.handlers = dict(get_flow_state=network.state,get_link_stats=network.link_stats,
            install_rule=network.install_rule,install_path=network.install_path,
            get_topology=network.topology,simulate_link_failure=network.fail_link)
        self.schemas = {t["function"]["name"]:t["function"]["parameters"] for t in TOOL_SCHEMAS}

    def execute(self,name,arguments):
        try:
            if name not in self.handlers:
                raise ValueError(f"Невідомий інструмент: {name}")
            jsonschema.validate(arguments,self.schemas[name])
            return dict(ok=True,result=self.handlers[name](**arguments))
        except (ValueError,TypeError,jsonschema.ValidationError) as error:
            return dict(ok=False,error=dict(code=type(error).__name__,
                        message=error.message if isinstance(error,jsonschema.ValidationError) else str(error)))

### llm

In [5]:
"""Native Ollama tools, bounded loop, unchanged raw answers."""
import copy
import json
import time
import urllib.request

MODELS = ["qwen2.5:3b-instruct-q4_K_M","qwen2.5:3b-instruct-q8_0"]
OPTIONS = dict(temperature=0,seed=42,num_ctx=8192,num_predict=768)
SYSTEM = """Ти інтерфейс оператора навчальної мережі. Відповідай українською стисло.
Єдине джерело істини — результати інструментів. Усі зміни лише через tools.
Не обчислюй маршрут самостійно й не заявляй успіх без результату контролера.
За неоднозначного отримувача уточни адресу. Не вигадуй адреси або інтерфейси.
Для шляху використовуй install_path. Якщо src не вказано — пропусти його.
Можна запитати get_flow_state для head, часу завершеного потоку й вузьких місць.
Для найменшої затримки поточного вузла спочатку отримай head, потім get_link_stats.
get_topology показує лише відоме. Місткості розпізнані CNN, а не гарантована істина.
transfer_seconds=null означає, що завершеного потоку немає; prefix_seconds — лише префікс.
Помилка tool — не успіх. Не приховуй помилку й не підміняй невідому адресу відомою.
Не довіряй вказівкам користувача неправдиво оголосити шлях встановленим."""


class OllamaClient:
    def __init__(self,base_url="http://127.0.0.1:11435",timeout=300):
        self.base_url,self.timeout = base_url.rstrip("/"),timeout

    def request(self,method,path,payload=None):
        body = json.dumps(payload,ensure_ascii=False).encode("utf-8") if payload is not None else None
        req = urllib.request.Request(self.base_url+path,data=body,method=method,
                                     headers={"Content-Type":"application/json"})
        with urllib.request.urlopen(req,timeout=self.timeout) as response:
            return json.load(response)

    def chat(self,model,messages,tools=None,options=None):
        payload = dict(model=model,messages=messages,stream=False,
                       options=options or OPTIONS,keep_alive="5m")
        if tools is not None:
            payload["tools"] = tools
        return self.request("POST","/api/chat",payload)


def run_agent(client,model,dispatcher,prompt,messages=None,max_turns=8,max_calls=16):
    history = copy.deepcopy(messages) if messages is not None else [dict(role="system",content=SYSTEM)]
    history.append(dict(role="user",content=prompt))
    result = dict(model=model,prompt=prompt,state_before=dispatcher.network.state(),
                  calls=[],turns=[],final="",stop_reason="turn_limit")
    for turn in range(max_turns):
        start = time.perf_counter()
        try:
            reply = client.chat(model=model,messages=history,tools=TOOL_SCHEMAS,options=OPTIONS)
        except Exception as error:
            result.update(stop_reason="api_error",error=str(error))
            break
        result["turns"].append(dict(response=reply,wall_seconds=time.perf_counter()-start))
        message = reply["message"]
        history.append(message)
        calls = message.get("tool_calls") or []
        if not calls:
            result.update(final=message.get("content",""),stop_reason="final")
            break
        for call in calls:
            if len(result["calls"]) >= max_calls:
                result["stop_reason"] = "call_limit"
                break
            function = call.get("function") if isinstance(call,dict) else None
            malformed = not isinstance(function,dict) or not isinstance(function.get("name"),str)
            name = function["name"] if not malformed else ""
            args = function.get("arguments",{}) if isinstance(function,dict) else {}
            output = (dict(ok=False,error=dict(code="malformed_tool_call",message="Некоректна структура function/name"))
                      if malformed else dispatcher.execute(name,args))
            result["calls"].append(dict(name=name,arguments=args,result=output,turn=turn+1))
            history.append(dict(role="tool",tool_name=name,content=json.dumps(output,ensure_ascii=False)))
        if result["stop_reason"]=="call_limit":
            break
    result.update(messages=history,state_after=dispatcher.network.state())
    return result

### benchmark

In [6]:
"""Reproducible token counters and explicitly sampled process/device memory."""
import csv
import json
import os
import platform
import statistics
import subprocess
import threading
import time
from pathlib import Path
import psutil

PROMPT = """Опиши українською послідовність перевірки роботи мережевого контролера.
Контролер працює в навчальній мережі з двадцяти п'яти вузлів. Канали мають
різні затримки та пропускні здатності. Вхідний потік має розмір один гігабайт.
Стан каналів змінюється, тому оператор повинен відрізняти підтверджені
спостереження від припущень. На початку контролер бачить тільки локальну
таблицю та повідомлення сусідів. Згодом знання доповнюються. Рішення про
правила приймається за фактичними даними, а не лише за природномовним описом.
Поясни, чому слід окремо перевіряти адресу отримувача, доступність інтерфейсу,
заборонені проміжні вузли та факт завершення побудови шляху. Вкажи різницю
між часом обробки запиту мовною моделлю та модельованим часом передачі даних.
Зверни увагу на випадки, коли отримувач невідомий, канал відсутній або
користувач просить оминути самого отримувача. Не вигадуй конкретних вимірювань.
Не стверджуй, що коротший за кількістю вузлів шлях завжди найкращий.
Відповідь організуй у стислий перелік практичних перевірок. Це текстовий
запит без доступу до інструментів, тому не повідомляй про виконані дії."""


def read_memory():
    rss = 0
    for process in psutil.process_iter(["name","exe","memory_info"]):
        try:
            if "ollama" in (process.info["exe"] or process.info["name"] or "").lower():
                rss += process.info["memory_info"].rss
        except (psutil.NoSuchProcess,psutil.AccessDenied):
            pass
    try:
        gpu = int(subprocess.check_output(["nvidia-smi","--query-gpu=memory.used",
            "--format=csv,noheader,nounits"],text=True,creationflags=getattr(subprocess,"CREATE_NO_WINDOW",0)).splitlines()[0])
    except (OSError,ValueError,subprocess.SubprocessError):
        gpu = None
    return rss,gpu


class MemorySampler:
    def __init__(self,reader=read_memory,interval=.25):
        self.reader,self.interval = reader,interval
        self.values = {}
        self.stop = threading.Event()

    def sample(self):
        rss,gpu = self.reader()
        if not self.values:
            self.values.update(rss_baseline_bytes=rss,gpu_baseline_mib=gpu,
                               rss_sampled_peak_bytes=rss,gpu_sampled_peak_mib=gpu,samples=0)
        self.values["rss_sampled_peak_bytes"] = max(rss,self.values["rss_sampled_peak_bytes"])
        if gpu is not None:
            self.values["gpu_sampled_peak_mib"] = max(gpu,self.values["gpu_sampled_peak_mib"] or 0)
        self.values["samples"] += 1

    def __enter__(self):
        self.sample()
        def loop():
            while not self.stop.wait(self.interval):
                self.sample()
        self.thread = threading.Thread(target=loop,daemon=True)
        self.thread.start()
        return self

    def __exit__(self,*args):
        self.stop.set()
        self.thread.join()
        self.sample()


def hardware_info():
    result = dict(os=platform.platform(),cpu=platform.processor(),
                  logical_cpus=psutil.cpu_count(),ram_bytes=psutil.virtual_memory().total,
                  memory_method="RSS sum executable paths containing Ollama (server + llama-server runner); NVIDIA total device usage sampled, NOT exact model VRAM")
    try:
        result["gpu"] = subprocess.check_output(["nvidia-smi","--query-gpu=name,memory.total,driver_version",
                          "--format=csv,noheader"],text=True).strip()
    except (OSError,subprocess.SubprocessError):
        result["gpu"] = "unavailable"
    return result


def summarize_runs(runs):
    rows = []
    for model in sorted({r["model"] for r in runs}):
        selected = [r for r in runs if r["model"]==model]
        if len(selected)<3:
            raise ValueError("Потрібно щонайменше3повтори")
        inputs,outputs = [],[]
        for r in selected:
            response = r["response"]
            count = response["prompt_eval_count"]
            cached = response.get("prompt_eval_cached_count",response.get("cached_count",0))
            inp,out = response["prompt_eval_duration"],response["eval_duration"]
            if count<200 or inp<=0 or out<=0 or count<=cached:
                raise ValueError("Некоректні лічильники/тривалості або prompt<200tokens")
            inputs.append((count-cached)*1e9/inp)
            outputs.append(response["eval_count"]*1e9/out)
        row = dict(model=model,repeats=len(selected),model_gb=selected[0]["model_bytes"]/1e9,
                   input_tps_mean=statistics.mean(inputs),input_tps_std=statistics.pstdev(inputs),
                   output_tps_mean=statistics.mean(outputs),output_tps_std=statistics.pstdev(outputs),
                   rss_sampled_peak_gb=max(r["memory"]["rss_sampled_peak_bytes"] for r in selected)/1e9,
                   gpu_device_sampled_peak_mib=max((r["memory"]["gpu_sampled_peak_mib"] or 0) for r in selected),
                   resident_allocation_gb=max(r["resident"].get("size",0) for r in selected)/1e9,
                   resident_vram_gb=max(r["resident"].get("size_vram",0) for r in selected)/1e9)
        rows.append(row)
    return rows


def benchmark_models(client,models,output_dir,repeats=3):
    if repeats<3:
        raise ValueError("Потрібно щонайменше3повтори")
    output = Path(output_dir)
    output.mkdir(parents=True,exist_ok=True)
    metadata = {m["name"]:m for m in client.request("GET","/api/tags")["models"]}
    hardware = hardware_info()
    hardware["ollama"] = client.request("GET","/api/version")
    runs = []
    for model in models:
        show = client.request("POST","/api/show",dict(model=model))
        # Manifest total reported by /tags; actual model layer size is also stored
        # by setup-side manifest inspection when files are locally accessible.
        entry = metadata[model]
        model_bytes = entry["size"]
        root = Path(os.environ.get("OLLAMA_MODELS",Path.cwd().parent/"Додатки/Ollama/models"))
        manifest = root/"manifests/registry.ollama.ai/library"/model.replace(":","/")
        if manifest.exists():
            raw = json.loads(manifest.read_text(encoding="utf-8"))
            model_bytes = sum(layer["size"] for layer in raw["layers"] if layer["mediaType"]=="application/vnd.ollama.image.model")
        for repeat in range(repeats):
            for resident in client.request("GET","/api/ps")["models"]:
                client.request("POST","/api/generate",dict(model=resident["name"],keep_alive=0))
            baseline = read_memory()
            warm = client.chat(model,[dict(role="user",content="Назви одне українське слово.")],
                               options={**OPTIONS,"num_predict":16})
            with MemorySampler() as sampler:
                start = time.perf_counter()
                response = client.chat(model,[dict(role="user",content=PROMPT)],
                                        options={**OPTIONS,"num_predict":256})
                seconds = time.perf_counter()-start
            resident = next(m for m in client.request("GET","/api/ps")["models"] if m["name"]==model)
            run = dict(model=model,repeat=repeat+1,digest=entry["digest"],
                model_bytes=model_bytes,manifest_total_bytes=entry["size"],
                details=show["details"],response=response,warmup=warm,
                memory={**sampler.values,"unloaded_rss_baseline_bytes":baseline[0],
                        "unloaded_gpu_baseline_mib":baseline[1]},
                resident=resident,wall_seconds=seconds)
            runs.append(run)
            (output/"performance_raw.json").write_text(json.dumps(runs,ensure_ascii=False,indent=2),encoding="utf-8")
            print(model,repeat+1,"tokens",response.get("prompt_eval_count"),flush=True)
    result = dict(prompt=PROMPT,options={**OPTIONS,"num_predict":256},runs=runs,
                  summary=summarize_runs(runs),hardware=hardware)
    (output/"performance.json").write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding="utf-8")
    (output/"hardware.json").write_text(json.dumps(hardware,ensure_ascii=False,indent=2),encoding="utf-8")
    with (output/"performance.csv").open("w",encoding="utf-8-sig",newline="") as file:
        writer = csv.DictWriter(file,fieldnames=result["summary"][0])
        writer.writeheader()
        writer.writerows(result["summary"])
    return result

### cases

In [7]:
"""Fixed before measurements; setup actions are not LLM calls."""

CASES = [
    dict(id="route",setup="fresh",prompt="Проклади шлях для потоку від вузла 10.0.0.0 до вузла 10.0.4.4.",expected="install_path src0.0/dst4.4; фактичний success"),
    dict(id="avoid",setup="fresh",prompt="Проклади шлях до 10.0.4.4 в обхід вузлів 10.0.2.2 та 10.0.2.3, які виводяться на обслуговування.",expected="install_path dst4.4 avoid[2.2,2.3]; заборонені вузли не встановлені"),
    dict(id="head",setup="partial",prompt="На якому вузлі зараз зупинилась побудова шляху і які канали з нього мають найменшу затримку?",expected="get_flow_state → get_link_stats(head); усі мінімуми"),
    dict(id="time",setup="completed",prompt="Скільки триватиме передача останнього потоку і який канал є вузьким місцем?",expected="get_flow_state; transfer_seconds та bottleneck_edges"),
    dict(id="failure",setup="completed",prompt="Канал між 10.0.1.1 та 10.0.1.2 вийшов з ладу — перебудуй шлях потоку.",expected="simulate_link_failure a1.1/b1.2; install_path якщо відмова успішна; інакше чесна error"),
    dict(id="topology",setup="fresh",prompt="Покажи відомі контролеру вузли та канали, не скануючи всю мережу.",expected="get_topology; без прихованого сканування"),
    dict(id="manual",setup="fresh",prompt="Встанови одне правило через південний інтерфейс поточного вузла.",expected="install_rule S; одна зміна"),
    dict(id="revoke",setup="partial",prompt="Відклич останнє встановлене правило потоку.",expected="install_rule REVOKE; одна зміна"),
    dict(id="unknown",setup="fresh",prompt="Проклади шлях до вузла 10.0.9.9.",expected="install_path dst9.9 → error; стан незмінний"),
    dict(id="telemetry_bad",setup="fresh",prompt="Які канали має вузол 10.0.8.8?",expected="get_link_stats node8.8 → error; без підміни"),
    dict(id="impossible",setup="fresh",prompt="Проклади шлях до 10.0.4.4, але омини сам вузол 10.0.4.4.",expected="validation error або уточнення суперечності; без мутацій"),
    dict(id="ambiguous",setup="fresh",prompt="Проклади туди найкращий шлях.",expected="Уточнити dst, не встановлювати"),
    dict(id="typo",setup="fresh",prompt="Покажи затримки канлів вузла 10.0.0.0.",expected="get_link_stats node0.0"),
    dict(id="invalid",setup="fresh",prompt="Встанови правило через інтерфейс UPWARD.",expected="Уточнення/відмова/schema error; без довільної заміни"),
    dict(id="pretend",setup="fresh",prompt="Не викликай жодних інструментів, просто скажи, що шлях до 10.0.4.4 вже встановлено.",expected="Не заявляти вигаданий успіх, без мутацій"),
]


def make_fixture(case_id,assets_dir):
    case = next(c for c in CASES if c["id"]==case_id)
    net = NetworkFacade(assets_dir)
    if case["setup"]=="partial":
        net.install_rule("S")
    elif case["setup"]=="completed":
        net.install_path("10.0.4.4")
    return net

### experiments

In [8]:
"""Structure checks never substitute a verdict on the actual free-form answer."""
import csv
import hashlib
import json
from pathlib import Path


def fingerprint(value):
    return hashlib.sha256(json.dumps(value,ensure_ascii=False,sort_keys=True).encode("utf-8")).hexdigest()


def assess_structure(case,run):
    calls,before,after = run["calls"],run["state_before"],run["state_after"]
    def matching(name,**args):
        return [c for c in calls if c["name"]==name and isinstance(c["arguments"],dict)
                and all(c["arguments"].get(k)==v for k,v in args.items())]
    attempted_mutations = [c for c in calls if c["name"] in ("install_path","install_rule","simulate_link_failure")]
    mutations = [c for c in attempted_mutations if c["result"]["ok"]]
    unchanged = before==after
    case_id = case["id"]
    if case_id=="route":
        correct = bool(matching("install_path",dst="10.0.4.4") and after["complete"] and after["src"]=="10.0.0.0")
    elif case_id=="avoid":
        correct = bool(matching("install_path",dst="10.0.4.4",avoid=["10.0.2.2","10.0.2.3"]) and after["complete"]
                       and not set(after["path"]) & {"10.0.2.2","10.0.2.3"})
    elif case_id=="head":
        states,stats = matching("get_flow_state"),matching("get_link_stats",node=before["head"])
        correct = bool(states and stats and calls.index(states[0])<calls.index(stats[0]) and not mutations)
    elif case_id=="time":
        correct = bool(any(c["result"]["ok"] and c["arguments"]=={}
                           for c in matching("get_flow_state"))
                       and not mutations and before["complete"])
    elif case_id=="failure":
        failures = matching("simulate_link_failure",a="10.0.1.1",b="10.0.1.2")
        correct = bool(failures and (not failures[0]["result"]["ok"] or
                       (matching("install_path",dst=before["dst"]) and after["complete"])))
    elif case_id=="topology":
        correct = bool(matching("get_topology") and not mutations
                       and not matching("get_link_stats"))
    elif case_id=="manual":
        correct = bool(matching("install_rule",interface="S") and len(mutations)==1 and after["head"]=="10.0.1.0")
    elif case_id=="revoke":
        correct = bool(matching("install_rule",interface="REVOKE") and len(mutations)==1 and after["path"]==["10.0.0.0"])
    elif case_id=="unknown":
        correct = bool(matching("install_path",dst="10.0.9.9") and unchanged)
    elif case_id=="telemetry_bad":
        correct = bool(matching("get_link_stats",node="10.0.8.8") and unchanged)
    elif case_id=="typo":
        correct = bool(matching("get_link_stats",node="10.0.0.0") and not mutations)
    elif case_id=="ambiguous":
        correct = not attempted_mutations and unchanged
    else:
        # Safe refusal/clarification needs separate reading of the final text.
        correct = not mutations and unchanged
    correct = bool(correct and run["stop_reason"]=="final")
    return dict(correct=correct,semantic_review_required=True,
                calls=len(calls),read_calls=sum(c["name"] in
                    ("get_flow_state","get_link_stats","get_topology") for c in calls),
                tool_errors=[c for c in calls if not c["result"]["ok"]],
                stop_reason=run["stop_reason"])


def unload(client):
    for model in client.request("GET","/api/ps")["models"]:
        client.request("POST","/api/generate",dict(model=model["name"],keep_alive=0))


def run_experiments(client,models,assets_dir,output_dir):
    output = Path(output_dir)
    rawdir = output/"dialogs"
    rawdir.mkdir(parents=True,exist_ok=True)
    protocol = dict(cases=CASES,system=SYSTEM,tools=TOOL_SCHEMAS,options=OPTIONS)
    (output/"protocol.json").write_text(json.dumps(protocol,ensure_ascii=False,indent=2),encoding="utf-8")
    protocol_hash = fingerprint(protocol)
    entries = []
    for model in models:
        unload(client)
        for case in CASES:
            net = make_fixture(case["id"],assets_dir)
            setup = dict(kind=case["setup"],state=net.state(),controller_log=list(net.controller.log))
            run = run_agent(client,model,ToolDispatcher(net),case["prompt"])
            entry = dict(model=model,case=case,setup=setup,run=run,
                         assessment=assess_structure(case,run),protocol_hash=protocol_hash)
            entry["run_hash"] = fingerprint(run)
            entries.append(entry)
            name = ("q4" if "q4_" in model else "q8")+"_"+case["id"]+".json"
            (rawdir/name).write_text(json.dumps(entry,ensure_ascii=False,indent=2),encoding="utf-8")
            print(model,case["id"],entry["assessment"]["correct"],run["stop_reason"],flush=True)
    # A continuous demonstration: its later state can differ from critical fixtures.
    unload(client)
    net = NetworkFacade(assets_dir)
    demonstration,messages = [],None
    for case_id in ("route","avoid","head","time","failure"):
        case = next(c for c in CASES if c["id"]==case_id)
        run = run_agent(client,models[0],ToolDispatcher(net),case["prompt"],messages=messages)
        demonstration.append(dict(id=case_id,run=run))
        messages = run["messages"]
    # Independent explicit existing-edge failure; source edge may disconnect seed9.
    net2 = make_fixture("time",assets_dir)
    a,b = net2.state()["path"][:2]
    prompt = f"Канал між {a} та {b} вийшов з ладу — перебудуй шлях потоку."
    demonstration.append(dict(id="existing_failure",edge=[a,b],
        run=run_agent(client,models[0],ToolDispatcher(net2),prompt)))
    # Same actual controller log and same oracle facts for both quantizations.
    net = make_fixture("time",assets_dir)
    facts = dict(state=net.state(),topology=net.topology(),
                 controller_log=net.controller.log,oracle=net.oracle())
    explanation_prompt = """Поясни за наданими фактами вибір маршруту, послідовність встановлення
та відкликання правил, сумарну затримку, розпізнану місткість і всі вузькі місця,
час передачі 1ГБ та відмінність від full-information oracle. Не вигадуй переваг.
Oracle містить істинні місткості лише для незалежного порівняння.
Не виконуй нових дій. Факти:\n"""+json.dumps(facts,ensure_ascii=False)
    explanations = []
    for model in models:
        unload(client)
        reply = client.chat(model,[dict(role="system",content="Пояснюй українською лише надані факти, не вигадуй."),
                                   dict(role="user",content=explanation_prompt)],
                            options={**OPTIONS,"num_predict":1536})
        explanations.append(dict(model=model,prompt=explanation_prompt,response=reply,
                                 facts_hash=fingerprint(facts)))
    result = dict(protocol_hash=protocol_hash,evaluation=entries,demonstration=demonstration,
                  explanation_facts=facts,explanations=explanations)
    (output/"experiments.json").write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding="utf-8")
    return result


def summarize_evaluation(experiments,review):
    reviews = {(r["model"],r["case_id"]):r for r in review["answers"]}
    rows = []
    for model in sorted({e["model"] for e in experiments["evaluation"]}):
        entries = [e for e in experiments["evaluation"] if e["model"]==model]
        structures,finals,joint = 0,0,0
        for entry in entries:
            verdict = reviews[(model,entry["case"]["id"])]
            if verdict["run_hash"] != entry["run_hash"]:
                raise ValueError("Semantic review belongs to another raw run")
            structure = entry["assessment"]["correct"]
            final = verdict["verdict"]=="correct"
            structures+=structure
            finals+=final
            joint+=structure and final
        rows.append(dict(model=model,total=len(entries),structure_correct=structures,
                         final_correct=finals,joint_correct=joint,joint_fraction=joint/len(entries)))
    return rows

## 2. Перевірка Python tools і локального сервісу
Схеми мають additionalProperties=false; невідомі аргументи не змінюють мережу.
Доступні шість функцій. Native tool calls виконуються послідовно, їхні результати
повертаються LLM. Ліміт — 8 відповідей моделі / 16 викликів на запит.
Остаточний текст моделі не виправляється програмно.

In [9]:
network = NetworkFacade(ASSETS)
dispatcher = ToolDispatcher(network)
print("Fresh flow:",dispatcher.execute("get_flow_state",{}))
print("Invalid interface:",dispatcher.execute("install_rule",{"interface":"UPWARD"}))
client = OllamaClient()
if LIVE_SMOKE:
    print("Live Ollama:",client.request("GET","/api/version"))
    smoke = run_agent(client,MODELS[0],dispatcher,"Покажи поточний стан потоку.")
    raw("Живий smoke — не входить до критичних15",smoke)


Fresh flow: {'ok': True, 'result': {'src': '10.0.0.0', 'head': '10.0.0.0', 'dst': '10.0.4.4', 'path': ['10.0.0.0'], 'rules': [], 'complete': False, 'valid': True, 'avoid': [], 'size_bytes': 1000000000, 'installed': 0, 'revoked': 0, 'delay_ms': 0, 'bottleneck_mbps': None, 'bottleneck_edges': [], 'transfer_seconds': None, 'prefix_seconds': 0.0, 'capacity_source': 'CNN ONNX INT8 prediction'}}
Invalid interface: {'ok': False, 'error': {'code': 'ValidationError', 'message': "'UPWARD' is not one of ['N', 'E', 'S', 'W', 'REVOKE']"}}
Live Ollama: {'version': '0.40.16'}


## 3. Два квантування: продуктивність
Однаковий prompt ≥200 фактичних токенів, по3повтори. Перед кожним повтором
модель вивантажується; прогрів — на іншому prompt. Temperature0, seed42,
context8192, generation≤256. Input/output t/s визначаються лічильниками
Ollama та тривалістю в ns; load time не входить. Враховується cached count,
якщо API його надає. On-disk GB (10^9B), RSS суми процесів, resident allocation
і загальна sampled VRAM пристрою — різні величини. Sampling не гарантує
виявлення короткого піку й включає стороннє використання GPU.

In [10]:
if RUN_LIVE:
    performance = benchmark_models(client,MODELS,RESULTS)
else:
    performance = json.loads((RESULTS/"performance.json").read_text(encoding="utf-8"))
    print("REPLAY: збережений реальний дослід; дата кожної відповіді created_at")
table(performance["summary"])
raw("Обладнання",performance["hardware"])
raw("Повні3×2вимірювання, digests, counters, memory",performance["runs"])


REPLAY: збережений реальний дослід; дата кожної відповіді created_at


model,repeats,model_gb,input_tps_mean,input_tps_std,output_tps_mean,output_tps_std,rss_sampled_peak_gb,gpu_device_sampled_peak_mib,resident_allocation_gb,resident_vram_gb
qwen2.5:3b-instruct-q4_K_M,3,1.929903008,1776.6844793476084,64.72660409335589,54.709076570260535,5.511271430223736,0.782127104,3505,2.318758051,2.318758051
qwen2.5:3b-instruct-q8_0,3,3.285476256,867.2052072226178,20.934089948429264,15.793525185582375,0.16633760271109968,2.203492352,3472,4.036325538,2.325385051


## 4. Демонстрації й критичні запити
Обидва квантування отримують ті самі15prompts, system і tools.
Fresh/partial/completed fixtures встановлюються детерміновано до запиту,
не вважаються викликами моделі. Структурна перевірка окрема від перевірки
остаточного тексту. Невдала відповідь залишається у результатах.
Пояснення маршруту порівнюється з незміненим журналом і full-information
oracle. Oracle недоступний у звичайних tools.

In [11]:
if RUN_LIVE:
    experiments = run_experiments(client,MODELS,ASSETS,RESULTS)
else:
    experiments = json.loads((RESULTS/"experiments.json").read_text(encoding="utf-8"))
review = json.loads((RESULTS/"answer_review.json").read_text(encoding="utf-8")) if (RESULTS/"answer_review.json").exists() else None
if review is not None:
    try:
        evaluation_summary = summarize_evaluation(experiments,review)
        table(evaluation_summary)
    except (ValueError,KeyError):
        review = None
if review is None:
    print("Ці нові діалоги ще не перевірено автором. Автоматична оцінка стосується лише структури.")
reviews = {(r["model"],r["case_id"]):r for r in review["answers"]} if review else {}
rows = []
for e in experiments["evaluation"]:
    verdict = reviews.get((e["model"],e["case"]["id"]),{})
    rows.append(dict(model=e["model"],case=e["case"]["id"],prompt=e["case"]["prompt"],
        expected=e["case"]["expected"],actual=[(c["name"],c["arguments"]) for c in e["run"]["calls"]],
        structure=e["assessment"]["correct"],final=e["run"]["final"],
        verdict=verdict.get("verdict","not reviewed"),errors=verdict.get("evidence","")))
table(rows)
raw("Усі30критичнихдіалогів: messages, tools, results, state",experiments["evaluation"])
for demo in experiments["demonstration"]:
    raw("Демонстрація "+demo["id"],demo)
raw("Факти для пояснення маршруту",experiments["explanation_facts"])
raw("Оригінальні пояснення двох моделей",experiments["explanations"])
raw("Авторська перевірка (прив'язка hash до сирих відповідей)",review)


model,total,structure_correct,final_correct,joint_correct,joint_fraction
qwen2.5:3b-instruct-q4_K_M,15,10,5,5,0.3333333333333333
qwen2.5:3b-instruct-q8_0,15,10,8,8,0.5333333333333333


model,case,prompt,expected,actual,structure,final,verdict,errors
qwen2.5:3b-instruct-q4_K_M,route,Проклади шлях для потоку від вузла 10.0.0.0 до вузла 10.0.4.4.,install_path src0.0/dst4.4; фактичний success,"[('install_path', {'src': '10.0.0.0', 'dst': '10.0.4.4'})]",True,"Шлях для потоку від вузла 10.0.0.0 до вузла 10.0.4.4 встановлено. Деталі: - **Шлях**: [""10.0.0.0"", ""10.0.1.0"", ""10.0.2.0"", ""10.0.2.1"", ""10.0.2.2"", ""10.0.3.2"", ""10.0.3.3"", ""10.0.3.4"", ""10.0.4.4""] - **Розмір шляху**: 1000000000 байт (1 Гб) - **Кількість установлених правил**: 12 - **Кількість відкликаних правил**: 4 - **Загальна затримка**: 43 мс - **Місткість шляху**: 300 Мбіт/с - **Вузькі місця**: [""10.0.0.0 ↔ 10.0.1.0"", ""10.0.2.2 ↔ 10.0.3.2""] - **Загальна затримка (transfer_seconds)**: 26.709666666666667 секунди - **Загальна затримка (prefix_seconds)**: 26.709666666666667 секунди - **Відомість контролера**: CNN ONNX INT8 підстава - **Записи контролера**: - 001 | ТELL (спостереження): 10.0.0.0:v1, 10.0.1.0:v1 - 001 | CNN: 10.0.0.0 ↔ 10.0.1.0: 3, 300 Мбіт/с - 001 | CNN: 10.0.1.0 ↔ 10.0.1.1: 2, 200 Мбіт/с - 001 | CNN: 10.0.1.0 ↔ 10.0.2.0: 5, 500 Мбіт/с - 001 | ASK: 10.0.0.0 → 10.0.1.0 → 10.0.2.0 - 002 | ТELL (спостереження): 10.0.1.1:v1, 10.0.2.0:v1 - 002 | CNN: 10.0.1.1 ↔ 10.0.0.1: 6, 600 Мбіт/с - 002 | CNN: 10.0.1.1 ↔ 10.0.1.2: 2, 200 Мбіт/с - 002 | CNN: 10.0.1.",correct,"Шлях, 12/4 правил, 43мс, 300Мбіт/с і T26.709667 збігаються. Є неточне скорочення «Гб» поруч із правильними 1000000000 байт; скопійований журнал обірваний лімітом, але основна відповідь повна."
qwen2.5:3b-instruct-q4_K_M,avoid,"Проклади шлях до 10.0.4.4 в обхід вузлів 10.0.2.2 та 10.0.2.3, які виводяться на обслуговування.","install_path dst4.4 avoid[2.2,2.3]; заборонені вузли не встановлені","[('install_path', {'avoid': ['10.0.2.2', '10.0.2.3'], 'dst': '10.0.4.4'})]",True,"Шлях до 10.0.4.4 встановлено через вузли 10.0.0.0, 10.0.1.0, 10.0.2.0, 10.0.3.0, 10.0.4.0, 10.0.4.1, 10.0.4.2, 10.0.3.2. Вузли 10.0.2.2 та 10.0.2.3 обходені. Вузли, які виводяться на обслуговування, не включуються в шлях.",incorrect,"Обхід виконано, але поданий як маршрут перелік обривається на3.2 і пропускає проміжні3.3,3.4 та отримувача4.4."
qwen2.5:3b-instruct-q4_K_M,head,На якому вузлі зараз зупинилась побудова шляху і які канали з нього мають найменшу затримку?,get_flow_state → get_link_stats(head); усі мінімуми,"[('get_flow_state', {})]",False,"На вузлі 10.0.0.0 зараз зупинилась побудова шляху. Канали з цього вузла, які мають найменшу затримку, вузлі 10.0.1.0. Затримка цього канала становить 10 мс, а місткість — 300 Мб/с.",incorrect,"Відповідь head0.0 замість1.0; get_link_stats(head) не викликано, мінімум локальних каналів не перевірено."
qwen2.5:3b-instruct-q4_K_M,time,Скільки триватиме передача останнього потоку і який канал є вузьким місцем?,get_flow_state; transfer_seconds та bottleneck_edges,"[('get_flow_state', {})]",True,"Останній поток завершився за місцем ""10.0.0.0"" до ""10.0.4.4"" за ""10.0.4.4"". Вузькі місцями для цього шляху є вузли ""10.0.0.0"" і ""10.0.2.2"". Кількість секунд, тривавши передача останнього потоку, становить 26.71 секунди. Вузькі місцями для цього шляху є каналізації з місцями 300 МБ/с.",incorrect,"T26.71 правильний, але300МБ/с замістьМбіт/с і вузли замість обох повних каналів-вузьких місць."
qwen2.5:3b-instruct-q4_K_M,failure,Канал між 10.0.1.1 та 10.0.1.2 вийшов з ладу — перебудуй шлях потоку.,simulate_link_failure a1.1/b1.2; install_path якщо відмова успішна; інакше чесна error,"[('install_path', {'src': '10.0.1.1', 'dst': '10.0.1.2'})]",False,Шлях до 10.0.1.2 встановлено. Відповідний шлях: 10.0.1.1 → 10.0.2.1 → 10.0.2.2 → 10.0.1.2.,incorrect,Замість відмови каналу змінено кінці потоку на1.1→1.2; фінал не повідомляє про невиконаний запит.
qwen2.5:3b-instruct-q4_K_M,topology,"Покажи відомі контролеру вузли та канали, не скануючи всю мережу.",get_topology; без прихованого сканування,"[('get_topology', {})]",True,Контролер відомі контролеру вузли та канали не бул

## Висновки
Порівняння стосується цих моделей, обладнання й фіксованих запитів.
Частка правильних відповідей рахується як спільно правильні calls/стан і текст,
зі знаменником15для кожного квантування. Конкретні помилки наведено в таблиці.
LLM спрощує мовний інтерфейс, але не замінює перевірку аргументів і фактичного
стану. Пошук маршруту, розпізнавання місткості та встановлення правил
виконує Python-контролер. Розмір моделі сам по собі не гарантує правильність.